In [37]:
# TASK 2: Data Engineering & Preparation
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

# Simple load (notebook runs from notebooks/ directory)
raw_df = pd.read_csv("../data/data-1774253631395.csv")
print(f"Loaded raw_df -> {raw_df.shape}")

Loaded raw_df -> (12231, 37)


## Task 2 Scope (Data Engineering & Preparation)
This notebook is intentionally limited to raw-data preparation using data-1774253631395.csv.

Included steps:
- Load raw dataset
- Clean missing and noisy data
- Feature engineering
- Data reshaping (pivot and aggregation)
- Document assumptions

In [38]:
# Initial structure check (raw dataset only)
print("raw_df columns:", raw_df.shape[1])
display(raw_df.head(3))

print("\nRaw dtypes (sample):")
display(raw_df.dtypes.head(15))

# Base table for cleaning pipeline
df = raw_df.copy()
print("Working DataFrame shape:", df.shape)

raw_df columns: 37


,case_number,court_identifier,case_status,category,section,priority,plaintiff,defendant,case_type,registration_date_bs,registration_date_ad,reg_year,reg_month,reg_day_of_week,verdict_date_ad,verdict_date_bs,verdict_judge,days_to_verdict,verdict_status,total_hearings,first_hearing_date,last_hearing_date,hearing_span_days,case_duration_days,avg_days_between_hearings,common_bench_type,common_decision_type,hearing_case_status,distinct_judges,distinct_lawyers,has_remarks,total_entities,distinct_sides,entity_sides,plaintiff_name,defendant_name,distinct_addresses
0,082-OA-0760,special,NaN,NaN,NaN,NaN,प्रमिला पोखरेल,नेपाल सरकार,उपस्थित हुने निवेदन,2082-12-06,2026-03-20,2026,3,5,NaN,NaN,NaN,NaN,Pending,1,2026-03-20,2026-03-20,0,0,NaN,8952,तारेखमा नराख्ने,अन्तिम आदेश,1,0,False,0,0,NaN,NaN,NaN,0
1,082-OA-0757,special,NaN,फाँट ख,NaN,NaN,हरिश कुमार सिंह समेत २,नेपाल सरकार,उपस्थित हुने निवेदन,2082-12-06,2026-03-20,2026,3,5,NaN,NaN,NaN,NaN,Pending,1,2026-03-20,2026-03-20,0,0,NaN,8952,थुनछेक आदेश (धरौटी),अन्तिम आदेश,1,0,True,0,0,NaN,NaN,NaN,0
2,082-OA-0758,special,NaN,फाँट ख,NaN,NaN,सन्जिव चालिसे,नेपाल सरकार,उपस्थित हुने निवेदन,2082-12-06,2026-03-20,2026,3,5,NaN,NaN,NaN,NaN,Pending,1,2026-03-20,2026-03-20,0,0,NaN,8952,तारेखमा नराख्ने,अन्तिम आदेश,1,0,True,0,0,NaN,NaN,NaN,0



Raw dtypes (sample):


case_number                 str
court_identifier            str
case_status                 str
category                    str
section                 float64
priority                float64
plaintiff                   str
defendant                   str
case_type                   str
registration_date_bs        str
registration_date_ad        str
reg_year                  int64
reg_month                 int64
reg_day_of_week           int64
verdict_date_ad         float64
dtype: object

Working DataFrame shape: (12231, 37)


In [39]:
# Cleaning: missing values, noisy text, duplicates, data types
df_clean = df.copy()

# 1) Standardize column names
df_clean.columns = (
    df_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[^0-9a-zA-Z]+", "_", regex=True)
    .str.replace(r"_+", "_", regex=True)
    .str.strip("_")
)

# 2) Normalize blank-like values to NaN
for col in df_clean.select_dtypes(include=["object"]).columns:
    df_clean[col] = (
        df_clean[col]
        .astype(str)
        .str.strip()
        .replace({"": np.nan, "nan": np.nan, "none": np.nan, "null": np.nan, "NULL": np.nan})
    )

# 3) Missing value handling
missing_pct = (df_clean.isna().mean() * 100).sort_values(ascending=False)
drop_cols = missing_pct[missing_pct > 70].index.tolist()
if drop_cols:
    df_clean = df_clean.drop(columns=drop_cols)
print("Dropped columns (>70% missing):", drop_cols if drop_cols else "None")

num_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df_clean.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

for col in num_cols:
    if df_clean[col].isna().any():
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())

for col in cat_cols:
    if df_clean[col].isna().any():
        mode_vals = df_clean[col].mode(dropna=True)
        fill_value = mode_vals.iloc[0] if not mode_vals.empty else "Unknown"
        df_clean[col] = df_clean[col].fillna(fill_value)

# 4) Type fixes: datetime and numeric-like object
date_cols = [c for c in df_clean.columns if "date" in c or c.endswith("_ad") or c.endswith("_bs")]
for col in date_cols:
    parsed = pd.to_datetime(df_clean[col], errors="coerce")
    if parsed.notna().mean() >= 0.5:
        df_clean[col] = parsed

for col in df_clean.select_dtypes(include=["object"]).columns:
    num_try = pd.to_numeric(df_clean[col], errors="coerce")
    if num_try.notna().mean() >= 0.9:
        df_clean[col] = num_try

# 5) Remove duplicates
dup_count = int(df_clean.duplicated().sum())
df_clean = df_clean.drop_duplicates().reset_index(drop=True)
print("Duplicates removed:", dup_count)

print("Cleaned shape:", df_clean.shape)
display(df_clean.head(3))

Dropped columns (>70% missing): ['section', 'verdict_date_ad', 'verdict_date_bs', 'verdict_judge', 'priority', 'days_to_verdict', 'avg_days_between_hearings']
Duplicates removed: 0
Cleaned shape: (12231, 30)


,case_number,court_identifier,case_status,category,plaintiff,defendant,case_type,registration_date_bs,registration_date_ad,reg_year,reg_month,reg_day_of_week,verdict_status,total_hearings,first_hearing_date,last_hearing_date,hearing_span_days,case_duration_days,common_bench_type,common_decision_type,hearing_case_status,distinct_judges,distinct_lawyers,has_remarks,total_entities,distinct_sides,entity_sides,plaintiff_name,defendant_name,distinct_addresses
0,082-OA-0760,special,चलिरहेको,उपस्थित हुने निवेदन,प्रमिला पोखरेल,नेपाल सरकार,उपस्थित हुने निवेदन,2082-12-06,2026-03-20,2026,3,5,Pending,1,2026-03-20,2026-03-20,0,0,8952,तारेखमा नराख्ने,अन्तिम आदेश,1,0,False,0,0,"defendant, plaintiff",नेपाल सरकार,नेपाल सरकार,0
1,082-OA-0757,special,चलिरहेको,फाँट ख,हरिश कुमार सिंह समेत २,नेपाल सरकार,उपस्थित हुने निवेदन,2082-12-06,2026-03-20,2026,3,5,Pending,1,2026-03-20,2026-03-20,0,0,8952,थुनछेक आदेश (धरौटी),अन्तिम आदेश,1,0,True,0,0,"defendant, plaintiff",नेपाल सरकार,नेपाल सरकार,0
2,082-OA-0758,special,चलिरहेको,फाँट ख,सन्जिव चालिसे,नेपाल सरकार,उपस्थित हुने निवेदन,2082-12-06,2026-03-20,2026,3,5,Pending,1,2026-03-20,2026-03-20,0,0,8952,तारेखमा नराख्ने,अन्तिम आदेश,1,0,True,0,0,"defendant, plaintiff",नेपाल सरकार,नेपाल सरकार,0


In [40]:
# Feature engineering
df_feat = df_clean.copy()

# A) Datetime-derived features
datetime_cols = df_feat.select_dtypes(include=["datetime64[ns]"]).columns.tolist()
for col in datetime_cols:
    df_feat[f"{col}_year"] = df_feat[col].dt.year
    df_feat[f"{col}_month"] = df_feat[col].dt.month
    df_feat[f"{col}_dayofweek"] = df_feat[col].dt.dayofweek

# B) Duration feature from available date pairs
if {"registration_date_ad", "verdict_date_ad"}.issubset(df_feat.columns):
    df_feat["duration_days"] = (df_feat["verdict_date_ad"] - df_feat["registration_date_ad"]).dt.days
elif {"first_hearing_date", "last_hearing_date"}.issubset(df_feat.columns):
    df_feat["duration_days"] = (df_feat["last_hearing_date"] - df_feat["first_hearing_date"]).dt.days

# C) Frequency/count features
if "plaintiff" in df_feat.columns:
    df_feat["plaintiff_case_count"] = df_feat["plaintiff"].map(df_feat["plaintiff"].value_counts())
if "defendant" in df_feat.columns:
    df_feat["defendant_case_count"] = df_feat["defendant"].map(df_feat["defendant"].value_counts())

# D) Simple business flag example
if "case_status" in df_feat.columns:
    df_feat["is_pending_case"] = df_feat["case_status"].astype(str).str.contains("Pending|चलिरहेको", case=False, regex=True).astype(int)

print("Feature engineered shape:", df_feat.shape)
display(df_feat.head(3))

Feature engineered shape: (12231, 46)


,case_number,court_identifier,case_status,category,plaintiff,defendant,case_type,registration_date_bs,registration_date_ad,reg_year,reg_month,reg_day_of_week,verdict_status,total_hearings,first_hearing_date,last_hearing_date,hearing_span_days,case_duration_days,common_bench_type,common_decision_type,hearing_case_status,distinct_judges,distinct_lawyers,has_remarks,total_entities,distinct_sides,entity_sides,plaintiff_name,defendant_name,distinct_addresses,registration_date_bs_year,registration_date_bs_month,registration_date_bs_dayofweek,registration_date_ad_year,registration_date_ad_month,registration_date_ad_dayofweek,first_hearing_date_year,first_hearing_date_month,first_hearing_date_dayofweek,last_hearing_date_year,last_hearing_date_month,last_hearing_date_dayofweek,duration_days,plaintiff_case_count,defendant_case_count,is_pending_case
0,082-OA-0760,special,चलिरहेको,उपस्थित हुने निवेदन,प्रमिला पोखरेल,नेपाल सरकार,उपस्थित हुने निवेदन,2082-12-06,2026-03-20,2026,3,5,Pending,1,2026-03-20,2026-03-20,0,0,8952,तारेखमा नराख्ने,अन्तिम आदेश,1,0,False,0,0,"defendant, plaintiff",नेपाल सरकार,नेपाल सरकार,0,2082.0,12.0,6.0,2026,3,4,2026,3,4,2026,3,4,0,1,8939,1
1,082-OA-0757,special,चलिरहेको,फाँट ख,हरिश कुमार सिंह समेत २,नेपाल सरकार,उपस्थित हुने निवेदन,2082-12-06,2026-03-20,2026,3,5,Pending,1,2026-03-20,2026-03-20,0,0,8952,थुनछेक आदेश (धरौटी),अन्तिम आदेश,1,0,True,0,0,"defendant, plaintiff",नेपाल सरकार,नेपाल सरकार,0,2082.0,12.0,6.0,2026,3,4,2026,3,4,2026,3,4,0,1,8939,1
2,082-OA-0758,special,चलिरहेको,फाँट ख,सन्जिव चालिसे,नेपाल सरकार,उपस्थित हुने निवेदन,2082-12-06,2026-03-20,2026,3,5,Pending,1,2026-03-20,2026-03-20,0,0,8952,तारेखमा नराख्ने,अन्तिम आदेश,1,0,True,0,0,"defendant, plaintiff",नेपाल सरकार,नेपाल सरकार,0,2082.0,12.0,6.0,2026,3,4,2026,3,4,2026,3,4,0,1,8939,1


In [41]:
# Data reshaping: pivot, aggregation, and long-format transformation
work = df_feat.copy()

# 1) Pivot table
index_col = "reg_month" if "reg_month" in work.columns else None
column_col = "case_status" if "case_status" in work.columns else None
value_col = None
for c in ["total_hearings", "case_duration_days", "duration_days"]:
    if c in work.columns and pd.api.types.is_numeric_dtype(work[c]):
        value_col = c
        break

if index_col and column_col and value_col:
    pivot_df = pd.pivot_table(
        work,
        index=index_col,
        columns=column_col,
        values=value_col,
        aggfunc="mean",
        fill_value=0,
    ).reset_index()
else:
    pivot_df = pd.DataFrame()

# 2) Aggregation table
group_col = "category" if "category" in work.columns else None
agg_cols = [c for c in ["total_hearings", "case_duration_days", "duration_days"] if c in work.columns and pd.api.types.is_numeric_dtype(work[c])]

if group_col and agg_cols:
    agg_dict = {c: ["count", "mean", "median", "max"] for c in agg_cols}
    agg_df = work.groupby(group_col, dropna=False).agg(agg_dict)
    agg_df.columns = ["_".join(col).strip() for col in agg_df.columns]
    agg_df = agg_df.reset_index()
else:
    agg_df = pd.DataFrame()

# 3) Long format (melt)
melt_id = [c for c in ["case_number", "category", "case_status"] if c in work.columns]
melt_values = [c for c in ["total_hearings", "case_duration_days", "duration_days"] if c in work.columns and pd.api.types.is_numeric_dtype(work[c])]

if melt_values:
    long_df = work[melt_id + melt_values].melt(
        id_vars=melt_id,
        value_vars=melt_values,
        var_name="metric",
        value_name="metric_value",
    )
else:
    long_df = pd.DataFrame()

print("pivot_df shape:", pivot_df.shape)
print("agg_df shape:", agg_df.shape)
print("long_df shape:", long_df.shape)
display(agg_df.head(5))

pivot_df shape: (12, 2908)
agg_df shape: (18, 13)
long_df shape: (36693, 5)


,category,total_hearings_count,total_hearings_mean,total_hearings_median,total_hearings_max,case_duration_days_count,case_duration_days_mean,case_duration_days_median,case_duration_days_max,duration_days_count,duration_days_mean,duration_days_median,duration_days_max
0,अदालतको अवहेलना,1,1.000000,1.0,1,1,1.000000,1.0,1,1,0.000000,0.0,0
1,अभिलेख,2,3.000000,3.0,4,2,93.500000,93.5,171,2,79.500000,79.5,146
2,इजलासमा पेश हुने निवेदन,2137,1.077211,1.0,4,2137,1.242396,0.0,225,2137,0.782405,0.0,224
3,उपस्थित हुने निवेदन,6198,1.072281,1.0,64,6198,2.682156,0.0,1949,6198,2.370119,0.0,1939
4,गैरकानुनी रुपमा जग्गाधनि दर्ता समेत,4,7.750000,7.5,10,4,250.250000,239.0,481,4,115.250000,86.5,247


## Assumptions Documented
1. Raw dataset (data/data-1774253631395.csv) is the system-of-record for cleaning.
2. This notebook performs Task 2 preparation on raw data only.
3. Columns with more than 70% missing values are dropped.
4. Numeric nulls are imputed with median; categorical nulls with mode (or Unknown if mode is empty).
5. Date columns are parsed only if at least 50% values are parseable.
6. Numeric coercion for text columns is applied only if at least 90% values parse as numeric.
7. Duplicate rows are removed after cleaning and type normalization.

In [42]:
# Additional validation checks (raw-clean pipeline)
validation = {
    "raw_rows": raw_df.shape[0],
    "raw_columns": raw_df.shape[1],
    "clean_rows": df_clean.shape[0],
    "clean_columns": df_clean.shape[1],
    "feature_rows": df_feat.shape[0],
    "feature_columns": df_feat.shape[1],
}

for k, v in validation.items():
    print(f"{k}: {v}")

print("\nTop columns by missing percentage after cleaning:")
display(((df_clean.isna().mean() * 100).sort_values(ascending=False).head(10)).to_frame("missing_pct"))

raw_rows: 12231
raw_columns: 37
clean_rows: 12231
clean_columns: 30
feature_rows: 12231
feature_columns: 46

Top columns by missing percentage after cleaning:


,missing_pct
registration_date_bs,1.422615
case_number,0.000000
case_status,0.000000
category,0.000000
plaintiff,0.000000
court_identifier,0.000000
defendant,0.000000
case_type,0.000000
registration_date_ad,0.000000
reg_year,0.000000


In [43]:
# Export Task 2 outputs
output_dir = Path("../data")
output_dir.mkdir(parents=True, exist_ok=True)

clean_path = output_dir / "task2_cleaned_dataset.csv"
feature_path = output_dir / "task2_feature_engineered_dataset.csv"
pivot_path = output_dir / "task2_pivot_table.csv"
agg_path = output_dir / "task2_aggregated_table.csv"
long_path = output_dir / "task2_long_format_table.csv"

df_clean.to_csv(clean_path, index=False)
df_feat.to_csv(feature_path, index=False)
if not pivot_df.empty:
    pivot_df.to_csv(pivot_path, index=False)
if not agg_df.empty:
    agg_df.to_csv(agg_path, index=False)
if not long_df.empty:
    long_df.to_csv(long_path, index=False)

print("Saved outputs:")
print("-", clean_path.resolve())
print("-", feature_path.resolve())
print("-", pivot_path.resolve(), "(if generated)")
print("-", agg_path.resolve(), "(if generated)")
print("-", long_path.resolve(), "(if generated)")

Saved outputs:
- C:\Users\HP\Desktop\corruption-data-visualization\data\task2_cleaned_dataset.csv
- C:\Users\HP\Desktop\corruption-data-visualization\data\task2_feature_engineered_dataset.csv
- C:\Users\HP\Desktop\corruption-data-visualization\data\task2_pivot_table.csv (if generated)
- C:\Users\HP\Desktop\corruption-data-visualization\data\task2_aggregated_table.csv (if generated)
- C:\Users\HP\Desktop\corruption-data-visualization\data\task2_long_format_table.csv (if generated)


In [44]:
# Quality checks for GitHub submission
qc = {
    "raw_shape": raw_df.shape,
    "clean_shape": df_clean.shape,
    "feature_shape": df_feat.shape,
    "duplicates_in_clean": int(df_clean.duplicated().sum()),
    "remaining_missing_cells": int(df_clean.isna().sum().sum()),
}

for k, v in qc.items():
    print(f"{k}: {v}")

missing_top = (df_clean.isna().mean() * 100).sort_values(ascending=False).head(10)
display(missing_top.to_frame("missing_pct"))

raw_shape: (12231, 37)
clean_shape: (12231, 30)
feature_shape: (12231, 46)
duplicates_in_clean: 0
remaining_missing_cells: 174


,missing_pct
registration_date_bs,1.422615
case_number,0.000000
case_status,0.000000
category,0.000000
plaintiff,0.000000
court_identifier,0.000000
defendant,0.000000
case_type,0.000000
registration_date_ad,0.000000
reg_year,0.000000


### Task 2 Rubric Alignment (Criterion 2: Data Engineering and Python Logic)
- Data loading: Raw source used from data/data-1774253631395.csv.
- Cleaning logic: Missing-value handling, duplicate checks, type normalization, and noise reduction performed.
- Feature engineering: Temporal, duration, frequency, and pending-status features created.
- Reshaping and transformation: Pivot table, aggregated summary table, and long-format table generated.
- Assumptions documented: parsing rules, fallback imputations, and high-missing-column treatment recorded in notebook narrative.
- Reproducibility: Exported outputs saved for downstream visualization and report integration.

In [46]:
# Final preview tables
print("Cleaned dataset preview")
display(df_clean.head(3))

print("Feature engineered dataset preview")
display(df_feat.head(3))

if not agg_df.empty:
    print("Aggregated table preview")
    display(agg_df.head(5))

Cleaned dataset preview


,case_number,court_identifier,case_status,category,plaintiff,defendant,case_type,registration_date_bs,registration_date_ad,reg_year,reg_month,reg_day_of_week,verdict_status,total_hearings,first_hearing_date,last_hearing_date,hearing_span_days,case_duration_days,common_bench_type,common_decision_type,hearing_case_status,distinct_judges,distinct_lawyers,has_remarks,total_entities,distinct_sides,entity_sides,plaintiff_name,defendant_name,distinct_addresses
0,082-OA-0760,special,चलिरहेको,उपस्थित हुने निवेदन,प्रमिला पोखरेल,नेपाल सरकार,उपस्थित हुने निवेदन,2082-12-06,2026-03-20,2026,3,5,Pending,1,2026-03-20,2026-03-20,0,0,8952,तारेखमा नराख्ने,अन्तिम आदेश,1,0,False,0,0,"defendant, plaintiff",नेपाल सरकार,नेपाल सरकार,0
1,082-OA-0757,special,चलिरहेको,फाँट ख,हरिश कुमार सिंह समेत २,नेपाल सरकार,उपस्थित हुने निवेदन,2082-12-06,2026-03-20,2026,3,5,Pending,1,2026-03-20,2026-03-20,0,0,8952,थुनछेक आदेश (धरौटी),अन्तिम आदेश,1,0,True,0,0,"defendant, plaintiff",नेपाल सरकार,नेपाल सरकार,0
2,082-OA-0758,special,चलिरहेको,फाँट ख,सन्जिव चालिसे,नेपाल सरकार,उपस्थित हुने निवेदन,2082-12-06,2026-03-20,2026,3,5,Pending,1,2026-03-20,2026-03-20,0,0,8952,तारेखमा नराख्ने,अन्तिम आदेश,1,0,True,0,0,"defendant, plaintiff",नेपाल सरकार,नेपाल सरकार,0


Feature engineered dataset preview


,case_number,court_identifier,case_status,category,plaintiff,defendant,case_type,registration_date_bs,registration_date_ad,reg_year,reg_month,reg_day_of_week,verdict_status,total_hearings,first_hearing_date,last_hearing_date,hearing_span_days,case_duration_days,common_bench_type,common_decision_type,hearing_case_status,distinct_judges,distinct_lawyers,has_remarks,total_entities,distinct_sides,entity_sides,plaintiff_name,defendant_name,distinct_addresses,registration_date_bs_year,registration_date_bs_month,registration_date_bs_dayofweek,registration_date_ad_year,registration_date_ad_month,registration_date_ad_dayofweek,first_hearing_date_year,first_hearing_date_month,first_hearing_date_dayofweek,last_hearing_date_year,last_hearing_date_month,last_hearing_date_dayofweek,duration_days,plaintiff_case_count,defendant_case_count,is_pending_case
0,082-OA-0760,special,चलिरहेको,उपस्थित हुने निवेदन,प्रमिला पोखरेल,नेपाल सरकार,उपस्थित हुने निवेदन,2082-12-06,2026-03-20,2026,3,5,Pending,1,2026-03-20,2026-03-20,0,0,8952,तारेखमा नराख्ने,अन्तिम आदेश,1,0,False,0,0,"defendant, plaintiff",नेपाल सरकार,नेपाल सरकार,0,2082.0,12.0,6.0,2026,3,4,2026,3,4,2026,3,4,0,1,8939,1
1,082-OA-0757,special,चलिरहेको,फाँट ख,हरिश कुमार सिंह समेत २,नेपाल सरकार,उपस्थित हुने निवेदन,2082-12-06,2026-03-20,2026,3,5,Pending,1,2026-03-20,2026-03-20,0,0,8952,थुनछेक आदेश (धरौटी),अन्तिम आदेश,1,0,True,0,0,"defendant, plaintiff",नेपाल सरकार,नेपाल सरकार,0,2082.0,12.0,6.0,2026,3,4,2026,3,4,2026,3,4,0,1,8939,1
2,082-OA-0758,special,चलिरहेको,फाँट ख,सन्जिव चालिसे,नेपाल सरकार,उपस्थित हुने निवेदन,2082-12-06,2026-03-20,2026,3,5,Pending,1,2026-03-20,2026-03-20,0,0,8952,तारेखमा नराख्ने,अन्तिम आदेश,1,0,True,0,0,"defendant, plaintiff",नेपाल सरकार,नेपाल सरकार,0,2082.0,12.0,6.0,2026,3,4,2026,3,4,2026,3,4,0,1,8939,1


Aggregated table preview


,category,total_hearings_count,total_hearings_mean,total_hearings_median,total_hearings_max,case_duration_days_count,case_duration_days_mean,case_duration_days_median,case_duration_days_max,duration_days_count,duration_days_mean,duration_days_median,duration_days_max
0,अदालतको अवहेलना,1,1.000000,1.0,1,1,1.000000,1.0,1,1,0.000000,0.0,0
1,अभिलेख,2,3.000000,3.0,4,2,93.500000,93.5,171,2,79.500000,79.5,146
2,इजलासमा पेश हुने निवेदन,2137,1.077211,1.0,4,2137,1.242396,0.0,225,2137,0.782405,0.0,224
3,उपस्थित हुने निवेदन,6198,1.072281,1.0,64,6198,2.682156,0.0,1949,6198,2.370119,0.0,1939
4,गैरकानुनी रुपमा जग्गाधनि दर्ता समेत,4,7.750000,7.5,10,4,250.250000,239.0,481,4,115.250000,86.5,247
